Определение орбиты методом ПВД(тест из эфемерид)

In [23]:
from math import *
import numpy as np

import pandas as pd

import os
import subprocess
from concurrent.futures import ThreadPoolExecutor

from astropy.time import Time

In [24]:
ASTEPH_DIR = '/Users/ymax/Documents/coding/claude/nea_orb/astEph' # локальная программа astEph с MPCORB.DAT
ASTEPH_RATE_STEP = 300.0    # с - шаг численного дифференцирования для скоростей из astEph
ASTID = '24445'
MU = 2.9591220828411956E-04 # а.е.^3/сут^2 - GM Солнца (как _GMS в astEph, = k^2)
C_AU_DAY = 299792.458*86400/1.495978707E+08 # скорость света, а.е./сут
EPS_J2000 = np.radians(23.43929111)         # наклон эклиптики J2000 (как в astEph)



N = 10
t = Time('2026-09-29T15:35:23.153', format='isot')
JD = float(t.copy(format='jd').value)+20*np.arange(10)/1440.0
JD

array([2461313.14957353, 2461313.16346242, 2461313.17735131,
       2461313.1912402 , 2461313.20512909, 2461313.21901797,
       2461313.23290686, 2461313.24679575, 2461313.26068464,
       2461313.27457353])

In [25]:
def _run_asteph(target, jd, location):
    """Один вызов astEph: (RA, DEC в градусах, V) на момент jd (UTC) для обсерватории location."""
    exe = os.path.join(ASTEPH_DIR, 'astEph')
    out = subprocess.run([exe, str(target), f'{jd:.8f}', f'obscode={location}'], cwd=ASTEPH_DIR,
                         capture_output=True, text=True, timeout=120)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {l.split('=')[0].strip(): l.split('=', 1)[1].split()[0] for l in out.stdout.splitlines() if '=' in l}
    return np.degrees(float(val['RA'])), np.degrees(float(val['DEC'])), float(val['Vmag'])

def get_asteph_ephemeris(target, JD, location, err=0.005):
    """Та же эфемерида, что get_horizons_ephemeris, но из локальной программы astEph (ASTEPH_DIR).
    astEph дает только положение и V, поэтому скорости считаются по положениям в моменты
    jd -+ ASTEPH_RATE_STEP (с), а DELTA не заполняется. Первый вызов делается отдельно: при новой
    эпохе MPCORB.DAT astEph сначала считает и сохраняет свой кэш интегрирования (~10 с). Остальные
    вызовы идут параллельно.
    err - стандартная ошибка положения (arcsec): к RA*cos(DEC) и DEC каждого эфемеридного положения
    добавляется независимый гауссов шум N(0, err). err=0 - без шума."""
    JD = np.atleast_1d(np.asarray(JD, dtype=float))
    if not os.path.isfile(os.path.join(ASTEPH_DIR, 'astEph')):
        raise FileNotFoundError(f'нет программы {ASTEPH_DIR}/astEph (сборка: cc -O2 -o astEph astEph.c -lm)')
    h = ASTEPH_RATE_STEP/86400
    jds = np.concatenate([JD, JD-h, JD+h])
    first = _run_asteph(target, jds[0], location)
    with ThreadPoolExecutor(max_workers=8) as ex:
        rest = list(ex.map(lambda jd: _run_asteph(target, jd, location), jds[1:]))
    res = np.array([first] + rest).reshape(3, len(JD), 3)
    (ra, dec, v), (ra1, dec1, _), (ra2, dec2, _) = [res[k].T for k in range(3)]
    dra = ((ra2-ra1+180) % 360 - 180)*np.cos(np.radians(dec))*3600/(2*ASTEPH_RATE_STEP/3600)
    ddec = (dec2-dec1)*3600/(2*ASTEPH_RATE_STEP/3600)
    if err > 0:
        ra = (ra + np.random.normal(0, err, len(JD))/3600/np.cos(np.radians(dec))) % 360
        dec = dec + np.random.normal(0, err, len(JD))/3600
    return pd.DataFrame(dict(JD=JD, RA_EPH=ra, DEC_EPH=dec, dRAcosD=dra, dDEC=ddec, # arcsec/hour
                             VMAG=v, DELTA=np.nan))


def get_observer_state(JD, location='217', asteph_dir=ASTEPH_DIR):
    """Гелиоцентрические положение R (а.е.), скорость V (а.е./сут) и ускорение A (а.е./сут^2)
    наблюдателя в обсерватории с кодом MPC location на момент JD (UTC), экваториальная система J2000.
    Считает программа astEph (режим --observer). Возвращает три numpy-массива по 3 компоненты."""
    out = subprocess.run([os.path.join(asteph_dir, 'astEph'), '--observer', f'{float(JD):.8f}',
                          f'obscode={location}'], cwd=asteph_dir, capture_output=True, text=True, timeout=300)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {k.strip(): v for k, v in (l.split('=', 1) for l in out.stdout.splitlines() if '=' in l)}
    R, V, A = (np.array(val[k].split(), dtype=float) for k in ('R', 'V', 'A'))
    return R, V, A

In [26]:
def analyze_vector_D(pos, n):
    # Вычисляем JD0
    jd0 = pos['JD'].mean()

    # Инициализируем массивы
    D = np.zeros(3)
    dotD = np.zeros(3)
    ddotD = np.zeros(3)

    # Вычисляем компоненты D
    pos['D0'] = np.cos(np.radians(pos['RA_EPH'])) * np.cos(np.radians(pos['DEC_EPH']))
    pos['D1'] = np.sin(np.radians(pos['RA_EPH'])) * np.cos(np.radians(pos['DEC_EPH']))
    pos['D2'] = np.sin(np.radians(pos['DEC_EPH']))

    # Инициализируем результаты
    results = []

    for j in range(3):
        DD = pos[f'D{j}']
        JD = pos['JD']

        # Вычисляем x = JD - JD0
        x = JD - jd0

        # Построение модели D = sum C_k * x^k
        X = np.column_stack([x**k for k in range(n+1)])

        # Метод наименьших квадратов
        C, residuals, rank, singular_values = np.linalg.lstsq(X, DD, rcond=None)

        D[j] = C[0]

        # Вычисляем первые и вторые производные в точке x=0
        if n >= 1:
            dotD[j] = C[1]
        else:
            dotD[j] = 0
        if n >= 2:
            ddotD[j] = 2 * C[2]
        else:
            ddotD[j] = 0
            
    return jd0, D, dotD, ddotD

In [28]:
pos = get_asteph_ephemeris(ASTID, JD, '217',err=0.005)
pos

,JD,RA_EPH,DEC_EPH,dRAcosD,dDEC,VMAG,DELTA
0,2.461313e+06,296.515012,11.887702,11.205227,-72.086603,16.49,NaN
1,2.461313e+06,296.516081,11.881023,11.318878,-72.128297,16.49,NaN
2,2.461313e+06,296.517153,11.874346,11.447465,-72.168295,16.49,NaN
3,2.461313e+06,296.518246,11.867663,11.590248,-72.206452,16.49,NaN
4,2.461313e+06,296.519347,11.860972,11.746326,-72.242304,16.49,NaN
5,2.461313e+06,296.520467,11.854284,11.914747,-72.275723,16.49,NaN
6,2.461313e+06,296.521605,11.847589,12.094450,-72.306512,16.49,NaN
7,2.461313e+06,296.522758,11.840896,12.284266,-72.334360,16.49,NaN
8,2.461313e+06,296.523929,11.834196,12.483006,-72.359317,16.49,NaN
9,2.461313e+06,296.525115,11.827494,12.689358,-72.381136,16.49,NaN


### Основные уравнения метода ПВД и необходимые соотношения

\begin{aligned}
D_x &= \cos\alpha \cos\delta \\
D_y &= \sin\alpha \cos\delta \\
D_z &= \sin\delta
\end{aligned}

\begin{aligned}
T_x &= -\sin\psi\,\sin\alpha - \cos\psi\,\sin\delta\,\cos\alpha, \\
T_y &= \sin\psi\,\cos\alpha - \cos\psi\,\sin\delta\,\sin\alpha, \\
T_z &= \cos\psi\,\cos\delta.
\end{aligned}

\begin{gather*}
\mathbf{D}\cdot\mathbf{D} = 1, \qquad
\mathbf{D}\cdot\dot{\mathbf{D}} = 0, \qquad
\dot{\mathbf{D}}\cdot\dot{\mathbf{D}} + \mathbf{D}\cdot\ddot{\mathbf{D}} = 0
\\[6pt]
\mu^2 = \bigl(\dot D\bigr)^2 = \dot D_x^2 + \dot D_y^2 + \dot D_z^2
\\[6pt]
\mu\cos\psi = \frac{\dot D_z}{\left(D_x^2 + D_y^2\right)^{1/2}}, \qquad
\mu\sin\psi = \frac{D_x \dot D_y - D_y \dot D_x}{\left(D_x^2 + D_y^2\right)^{1/2}}
\\[6pt]
\mu^3 \varkappa =
\begin{vmatrix}
D_x & D_y & D_z \\
\dot D_x & \dot D_y & \dot D_z \\
\ddot D_x & \ddot D_y & \ddot D_z
\end{vmatrix}
\end{gather*}

#### Уравнения метода ПВД
\begin{gather*}
d\,\mathbf{D} = \mathbf{r} - \mathbf{g}
\\[4pt]
\dot d\,\mathbf{D} + d\mu\,\mathbf{T} = \dot{\mathbf{r}} - \dot{\mathbf{g}}
\\[4pt]
\left(\ddot d - \mu^2 d + \kappa^2 r^{-3} d\right)\mathbf{D}
+ \left(2\mu\dot d + d\dot\mu\right)\mathbf{T}
+ \varkappa\mu^2 d\,\mathbf{M}
= -\kappa^2 r^{-3}\,\mathbf{g} - \ddot{\mathbf{g}}
\end{gather*}

#### Решение уравнений

Здесь $\mathbf{M} = \mathbf{D}\times\mathbf{T}$ (правая тройка $\mathbf{D}, \mathbf{T}, \mathbf{M}$), $(\mathbf{a}\,\mathbf{b}\,\mathbf{c}) = (\mathbf{a}\times\mathbf{b})\cdot\mathbf{c}$ - смешанное произведение, так что $(\mathbf{T}\,\mathbf{D}\,\mathbf{M}) = -1$. Третье уравнение - проекция уравнения для $\ddot{(d\mathbf{D})}$ на $\mathbf{M}$.

\begin{gather*}
\mathbf{r} = \mathbf{g} + d\,\mathbf{D}
\\[4pt]
r^2 = g^2 + d^2 + 2\,(\mathbf{g}\cdot\mathbf{D})\,d = C_0 + 2C_1 d + d^2
\\[4pt]
-\varkappa\mu^2 d\,\underbrace{(\mathbf{T}\,\mathbf{D}\,\mathbf{M})}_{-1}
= \varkappa\mu^2 d
= \kappa^2\,(\mathbf{T}\,\mathbf{D}\,\mathbf{g})\,r^{-3} + (\mathbf{T}\,\mathbf{D}\,\ddot{\mathbf{g}})
\\[4pt]
C d = C_3\, r^{-3} + C_2
\end{gather*}

Константы:

\begin{aligned}
C_0 &= g^2 = \mathbf{g}\cdot\mathbf{g}, \\
C_1 &= \mathbf{g}\cdot\mathbf{D}, \\
C   &= \varkappa\mu^2, \\
C_2 &= (\mathbf{T}\,\mathbf{D}\,\ddot{\mathbf{g}}), \\
C_3 &= \kappa^2\,(\mathbf{T}\,\mathbf{D}\,\mathbf{g}).
\end{aligned}

Еще раз уравнения:

\begin{gather*}
r^2 = C_0 + 2C_1 d + d^2
\\[4pt]
C d = C_3\, r^{-3} + C_2
\end{gather*}

In [29]:
JD0, D, dotD, ddotD = analyze_vector_D(pos, n=3)
D, JD0

(array([ 0.43698077, -0.87568577,  0.20548051]), np.float64(2461313.21207353))

In [30]:
g, dot_g, ddot_g = get_observer_state(JD0, location='217', asteph_dir=ASTEPH_DIR)
g, dot_g, ddot_g

(array([0.99585022, 0.09930656, 0.04307114]),
 array([-0.00207364,  0.01581567,  0.00677448]),
 array([-1.46940966e-03,  3.46533934e-04, -8.97035706e-06]))

In [31]:
def pvd_frame(D, dotD, ddotD):
    """Величины метода ПВД по D, dD/dt, d2D/dt2 (раздел «Основные уравнения метода ПВД»):
    mu = |dD/dt|, позиционный угол psi (рад), кривизна varkappa, единичные векторы T и M = D x T."""
    Dx, Dy, Dz = D
    alpha, delta = np.arctan2(Dy, Dx), np.arcsin(Dz)
    mu = np.sqrt(dotD @ dotD)                       # mu^2 = dD.dD
    rho = np.hypot(Dx, Dy)                          # (Dx^2 + Dy^2)^(1/2) = cos(delta)
    psi = np.arctan2((Dx*dotD[1] - Dy*dotD[0])/rho, dotD[2]/rho)  # mu sin(psi), mu cos(psi)
    T = np.array([-np.sin(psi)*np.sin(alpha) - np.cos(psi)*np.sin(delta)*np.cos(alpha),
                   np.sin(psi)*np.cos(alpha) - np.cos(psi)*np.sin(delta)*np.sin(alpha),
                   np.cos(psi)*np.cos(delta)])
    varkappa = np.linalg.det(np.array([D, dotD, ddotD]))/mu**3  # mu^3 varkappa = |D, dD, d2D|
    M = np.cross(D, T)
    return dict(mu=mu, psi=psi, varkappa=varkappa, T=T, M=M)

def laplace_coefficients(D, dotD, ddotD, g, ddot_g):
    """Коэффициенты уравнений метода ПВД  r^2 = C0 + 2 C1 d + d^2,  C d = C3 r^-3 + C2.
    Получены скалярным умножением уравнения для d2/dt2 (d D) на M = D x T (kappa^2 = MU):
    C = varkappa mu^2, C0 = g.g, C1 = g.D, C2 = (T D g'') = -M.g'', C3 = kappa^2 (T D g) = -kappa^2 M.g."""
    f = pvd_frame(D, dotD, ddotD)
    M = f['M']
    return dict(C=f['varkappa']*f['mu']**2, C0=g @ g, C1=g @ D, C2=-(M @ ddot_g), C3=-MU*(M @ g))

coef = laplace_coefficients(D, dotD, ddotD, g, ddot_g)
pd.Series(coef)

C     0.001325
C0    1.003435
C1    0.357056
C2    0.001114
C3   -0.000272
dtype: float64

In [32]:
D_MIN = 1e-4     # а.е. (~15000 км) - решения ближе к наблюдателю считаются ложными
ROOT_INDEX = 0   # какое из оставшихся решений брать, если их несколько

def solve_rd(C, C0, C1, C2, C3):
    """Все положительные вещественные корни уравнения Лагранжа и соответствующие d (из lagrange_rd.tex)."""
    al, be = C2 / C, C3 / C
    a = -(al**2 + 2*C1*al + C0)
    b = -2*be*(al + C1)
    c = -be**2
    roots = np.roots([1, 0, a, 0, 0, b, 0, 0, c]) # коэффициенты при r^8, r^7, ..., r^0
    sols = []
    for r in roots:
        if abs(r.imag) < 1e-10 and r.real > 0:
            r = r.real
            sols.append((r, al + be / r**3))
    return sorted(sols)

sols = solve_rd(**coef)
for r_, d_ in sols:
    print(f'r = {r_:.6f} а.е., d = {d_:+.6f} а.е.' + ('' if d_ > D_MIN else '  <- отброшено (d <= D_MIN)'))
good = [(r_, d_) for r_, d_ in sols if d_ > D_MIN]
if not good:
    raise RuntimeError('нет физического решения (d > D_MIN)')
r, d = good[min(ROOT_INDEX, len(good)-1)]
print(f'выбрано: r = {r:.6f} а.е., d = {d:.6f} а.е.')

r = 1.470004 а.е., d = +0.776507 а.е.
выбрано: r = 1.470004 а.е., d = 0.776507 а.е.


In [33]:
def laplace_state(D, dotD, ddotD, g, dot_g, ddot_g, r, d):
    """dd/dt и гелиоцентрические векторы положения и скорости астероида.
    dd/dt - из проекции уравнения для d2/dt2 (d D) на T:  2 mu dd/dt + d dmu/dt = -kappa^2 r^-3 g.T - g''.T,
    где dmu/dt = dD.d2D/mu. Положение r = g + d D, скорость v = g' + dd/dt D + d mu T."""
    f = pvd_frame(D, dotD, ddotD)
    mu, T = f['mu'], f['T']
    dot_mu = dotD @ ddotD/mu
    dot_d = -(MU/r**3*(g @ T) + ddot_g @ T + d*dot_mu)/(2*mu)
    return dot_d, g + d*D, dot_g + dot_d*D + d*mu*T

dot_d, r_vec, v_vec = laplace_state(D, dotD, ddotD, g, dot_g, ddot_g, r, d)
JD_EPOCH = JD0 - d/C_AU_DAY
print(f'dd/dt = {dot_d:+.6f} а.е./сут, световое запаздывание {d/C_AU_DAY*86400:.1f} с, эпоха JD {JD_EPOCH:.6f} UTC')
print('r =', r_vec, f' а.е. (|r| = {np.linalg.norm(r_vec):.6f}, из уравнения Лагранжа r = {r:.6f})')
print('v =', v_vec, ' а.е./сут')

dd/dt = -0.001805 а.е./сут, световое запаздывание 387.5 с, эпоха JD 2461313.207589 UTC
r = [ 1.33516879 -0.58066946  0.20262817]  а.е. (|r| = 1.470004, из уравнения Лагранжа r = 1.470004)
v = [-1.30655608e-03  1.66731064e-02  1.48636729e-05]  а.е./сут


In [34]:
def orbital_elements(r_eq, v_eq):
    """Гелиоцентрические элементы (эклиптика и равноденствие J2000) по векторам в экваториальной системе.
    a в а.е., углы в градусах; для гиперболы M = nan."""
    ce, se = np.cos(EPS_J2000), np.sin(EPS_J2000)
    Rot = np.array([[1, 0, 0], [0, ce, se], [0, -se, ce]])
    rv, vv = Rot @ r_eq, Rot @ v_eq
    rn = np.linalg.norm(rv)
    h = np.cross(rv, vv)
    a = 1/(2/rn - vv @ vv/MU)
    ev = np.cross(vv, h)/MU - rv/rn
    e = np.linalg.norm(ev)
    inc = degrees(acos(h[2]/np.linalg.norm(h)))
    node = np.cross([0, 0, 1], h)
    Om = degrees(atan2(node[1], node[0])) % 360
    w = degrees(acos(np.clip(node @ ev/np.linalg.norm(node)/e, -1, 1)))
    w = w if ev[2] >= 0 else 360 - w
    nu = acos(np.clip(ev @ rv/e/rn, -1, 1))
    nu = nu if rv @ vv >= 0 else 2*pi - nu
    if e < 1:
        E = 2*atan(sqrt((1-e)/(1+e))*tan(nu/2))
        M = degrees(E - e*sin(E)) % 360
    else:
        M = np.nan
    return dict(a=a, e=e, i=inc, Node=Om, Peri=w, M=M)

def _unpack_mpc_epoch(s):
    """Упакованная эпоха MPC (напр. K2669) -> JD (TT, 0h)."""
    dig = lambda c: int(c) if c.isdigit() else ord(c) - ord('A') + 10
    year = (ord(s[0]) - ord('A') + 10)*100 + int(s[1:3])
    return Time(f'{year:04d}-{dig(s[3]):02d}-{dig(s[4]):02d}', scale='tt').jd

def read_mpcorb(target, path=os.path.join(ASTEPH_DIR, 'MPCORB.DAT')):
    """Элементы астероида target из MPCORB.DAT (номер ищется как '(номер)' в читаемом обозначении)."""
    key = str(target).strip()
    with open(path, errors='replace') as f:
        for line in f:
            name = line[166:194].strip() if len(line) > 190 else ''
            if name and (name.startswith(f'({key})') or name == key or name.split(')')[-1].strip() == key):
                return dict(epoch=_unpack_mpc_epoch(line[20:25]), M=float(line[26:35]), Peri=float(line[37:46]),
                            Node=float(line[48:57]), i=float(line[59:68]), e=float(line[70:79]),
                            n=float(line[80:91]), a=float(line[92:103]), line=line.rstrip())
    raise KeyError(f'{target} не найден в {path}')

el = orbital_elements(r_vec, v_vec)
mpc = read_mpcorb(ASTID)
mpc_M = (mpc['M'] + mpc['n']*(JD_EPOCH - mpc['epoch'])) % 360 # перенос M на нашу эпоху
cmp = pd.DataFrame({'ПВД': el, 'MPCORB': {k: (mpc_M if k == 'M' else mpc[k]) for k in el}})
cmp['разность'] = cmp['ПВД'] - cmp['MPCORB']
print(f"эпоха JD {JD_EPOCH:.5f}; MPCORB: эпоха элементов JD {mpc['epoch']:.1f}, M перенесена на нашу эпоху")
cmp

эпоха JD 2461313.20759; MPCORB: эпоха элементов JD 2461200.5, M перенесена на нашу эпоху


,ПВД,MPCORB,разность
a,2.407731,2.210943,0.196789
e,0.578642,0.545700,0.032942
i,24.296291,23.769070,0.527221
Node,202.218312,203.134910,-0.916598
Peri,217.532361,219.122640,-1.590279
M,336.824149,333.065156,3.758992
